# Opinion-Leader Message Selection Mechanism Probe

- **Project:** Mass-communication opinion-leader model
- **Submodel ID and version:** `SM-OL-SELECT-01` v0.1
- **Framework link:** Reviewed `opleader` rough design
- **Probe type:** Mechanism probe with a fixed network boundary
- **Date:** 2026-09-04
- **Status:** Runnable

## 1. Question, Decision, and Framework Link

- **Primary question:** Does the proposed selection mechanism produce role-dependent press exposure and tie-dependent opinion-leader exposure without inspecting message stance?
- **Decision supported:** Retain this simplified selector as the V1 access-and-network mechanism for later paired coupling.
- **Shared interfaces:** `Message`, `Exposure`, `NetworkState`, `SelectionContext`, and `MessageSelection`; shared source and recipient role kinds are reused from `opinion_model.opleader`.
- **Highest intended claim level:** V1 isolated-submodel behavior under a fixed synthetic network and generated message pool.
- **Why a unit test alone is insufficient:** Unit tests verify individual rules; this probe also shows their joint exposure pattern and stochastic rates across recipient roles and network positions.

## 2. Provenance and Boundary Contract

- The selector is an assistant-constructed formalization of the researcher-approved simplification: press access is Bernoulli by recipient role, while opinion-leader messages travel through directed leader-to-ordinary ties.
- The numerical probabilities, fixed network, message stances, trial count, and seed are assumed experimental settings, not empirical estimates or calibrated values.
- The role-differentiated scenario uses `P(press→leader)=0.8` and `P(press→ordinary)=0.2`; the equal-rate scenario is retained as a source-access null.
- `NetworkState.neighbors_by_agent[recipient]` lists message producers that recipient can receive from. Leader-tie delivery is deterministic in V1.
- Press is an external originator and therefore bypasses the adaptive-agent network. The same press `Message` object can be delivered independently to several recipients.
- Press→leader, press→ordinary, and leader→ordinary are active. Leader→leader is consciously retained as an inactive relation rather than removed from the broader ontology.
- Production, belief initialization, aggregation, opinion updating, network updating, endogenous attention, and all feedbacks are omitted. Capacity is configured as nonbinding.
- Each trial repeats selection from the same fixed round-1 message pool; it is an independent exposure opportunity, not a temporal simulation round.

## 3. Expected Outcomes Before Running

- With press probability zero or one, observed press delivery is exactly zero or one for the affected recipient role.
- A press message can reach a recipient with no network ties.
- An ordinary recipient gets every leader message associated with an incoming leader tie and no message from an untied leader.
- A leader receives no leader message even if a leader-to-leader tie is placed in the fixed network.
- Flipping message stances while preserving source and message IDs leaves the selected IDs unchanged.
- In repeated trials, observed press rates fall within fixed four-standard-error tolerances of the configured role probabilities.
- Leader-message exposure counts equal the fixed number of incoming leader ties.
- Passing supports implementation consistency only; it does not validate the access probabilities, network, deterministic tie delivery, or any opinion outcome.

In [ ]:
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'src' / 'opinion_model').is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from opinion_model.core import Message, NetworkState, SelectionContext
from opinion_model.opleader import (
    OpinionLeaderMessageSelection,
    OriginatorKind,
    RecipientKind,
)

RUN = {
    'submodel_id': 'SM-OL-SELECT-01',
    'submodel_version': '0.1',
    'boundary_scenario': 'fixed directed network and generated round-1 pool',
    'trials_per_scenario': 5_000,
    'seed': 20260904,
    'leaders': [0, 1],
    'ordinary_agents': [2, 3, 4],
    'press_id': 10,
    'capacity': 10,
    'press_scenarios': {
        'equal-rate null': {'leader': 0.5, 'ordinary': 0.5},
        'role-differentiated': {'leader': 0.8, 'ordinary': 0.2},
    },
    'active_relations': [
        'press->leader', 'press->ordinary', 'leader->ordinary',
    ],
    'inactive_relations': ['leader->leader'],
    'active_feedbacks': [],
    'omitted_feedbacks': [
        'production', 'belief initialization', 'aggregation',
        'opinion updating', 'network updating', 'attention competition',
    ],
}
RUN

## 4. Minimal Specification and Implementation

For each press message and recipient `i`, delivery is an independent Bernoulli event with `p_press(role_i)`. For a leader message from `j`, delivery equals one only when `i` is ordinary and `j` appears in `neighbors_by_agent[i]`. No second Bernoulli draw is applied to an active leader tie. The selector imports the tested package implementation and returns `Exposure` objects; it does not aggregate messages or alter beliefs.

In [ ]:
LEADER_0, LEADER_1 = RUN['leaders']
ORDINARY_2, ORDINARY_3, ORDINARY_4 = RUN['ordinary_agents']
PRESS_ID = RUN['press_id']
RECIPIENT_KIND_BY_ID = {
    LEADER_0: RecipientKind.LEADER,
    LEADER_1: RecipientKind.LEADER,
    ORDINARY_2: RecipientKind.ORDINARY,
    ORDINARY_3: RecipientKind.ORDINARY,
    ORDINARY_4: RecipientKind.ORDINARY,
}
ORIGINATOR_KIND_BY_ID = {
    LEADER_0: OriginatorKind.LEADER,
    LEADER_1: OriginatorKind.LEADER,
    PRESS_ID: OriginatorKind.PRESS,
}
NETWORK = NetworkState({
    LEADER_0: (),
    LEADER_1: (),
    ORDINARY_2: (LEADER_0,),
    ORDINARY_3: (LEADER_0, LEADER_1),
    ORDINARY_4: (),
})
MESSAGE_POOL = (
    Message('r1:press10', 1, PRESS_ID, 1),
    Message('r1:a0', 1, LEADER_0, 1),
    Message('r1:a1', 1, LEADER_1, -1),
)
CONTEXT = SelectionContext(
    round_index=1, capacity=RUN['capacity'], exclude_self_messages=True
)

def make_selector(probabilities):
    return OpinionLeaderMessageSelection(
        originator_kind_by_id=ORIGINATOR_KIND_BY_ID,
        recipient_kind_by_id=RECIPIENT_KIND_BY_ID,
        press_delivery_probability_by_recipient_kind={
            RecipientKind.LEADER: probabilities['leader'],
            RecipientKind.ORDINARY: probabilities['ordinary'],
        },
    )

pd.DataFrame([
    {
        'recipient_id': recipient_id,
        'recipient_kind': RECIPIENT_KIND_BY_ID[recipient_id].value,
        'eligible_leaders': NETWORK.eligible_producers(recipient_id),
    }
    for recipient_id in RECIPIENT_KIND_BY_ID
] )

## 5. Deterministic and Boundary Checks

The following hand-calculable cases verify probability extremes, external press access, directed tie delivery, the inactive leader-to-leader relation, stance blindness, message identity, and the nonbinding-capacity boundary.

In [ ]:
zero_one_selector = make_selector({'leader': 1.0, 'ordinary': 0.0})
leader_exposures = zero_one_selector(
    LEADER_0, MESSAGE_POOL, NETWORK, CONTEXT, np.random.default_rng(1)
)
ordinary_2_exposures = zero_one_selector(
    ORDINARY_2, MESSAGE_POOL, NETWORK, CONTEXT, np.random.default_rng(2)
)
ordinary_3_exposures = zero_one_selector(
    ORDINARY_3, MESSAGE_POOL, NETWORK, CONTEXT, np.random.default_rng(3)
)
ordinary_4_exposures = zero_one_selector(
    ORDINARY_4, MESSAGE_POOL, NETWORK, CONTEXT, np.random.default_rng(4)
)
assert [item.message.producer_id for item in leader_exposures] == [PRESS_ID]
assert [item.message.producer_id for item in ordinary_2_exposures] == [LEADER_0]
assert [item.message.producer_id for item in ordinary_3_exposures] == [
    LEADER_0, LEADER_1
]
assert ordinary_4_exposures == ()

all_press_selector = make_selector({'leader': 1.0, 'ordinary': 1.0})
press_only = (MESSAGE_POOL[0],)
press_without_tie = all_press_selector(
    ORDINARY_4, press_only, NETWORK, CONTEXT, np.random.default_rng(5)
)
assert len(press_without_tie) == 1
assert press_without_tie[0].message is MESSAGE_POOL[0]

network_with_leader_tie = NetworkState({
    LEADER_0: (LEADER_1,),
    LEADER_1: (),
    ORDINARY_2: (LEADER_0,),
    ORDINARY_3: (LEADER_0, LEADER_1),
    ORDINARY_4: (),
})
leader_to_leader = all_press_selector(
    LEADER_0, (MESSAGE_POOL[2],), network_with_leader_tie, CONTEXT,
    np.random.default_rng(6),
)
assert leader_to_leader == ()

flipped_pool = tuple(
    Message(item.message_id, item.round_index, item.producer_id, -item.stance)
    for item in MESSAGE_POOL
)
original_ids = [
    item.message.message_id
    for item in all_press_selector(
        ORDINARY_3, MESSAGE_POOL, NETWORK, CONTEXT, np.random.default_rng(7)
    )
]
flipped_ids = [
    item.message.message_id
    for item in all_press_selector(
        ORDINARY_3, flipped_pool, NETWORK, CONTEXT, np.random.default_rng(7)
    )
]
assert original_ids == flipped_ids

try:
    all_press_selector(
        ORDINARY_3, MESSAGE_POOL, NETWORK, SelectionContext(1, 2, True),
        np.random.default_rng(8),
    )
except ValueError as error:
    assert 'attention competition is omitted' in str(error)
else:
    raise AssertionError('A binding capacity must be rejected in V1.')

boundary_checks = pd.DataFrame([
    {'case': 'press to leader, p=1', 'selected_ids': [leader_exposures[0].message.message_id]},
    {'case': 'ordinary 2, one leader tie', 'selected_ids': [item.message.message_id for item in ordinary_2_exposures]},
    {'case': 'ordinary 3, two leader ties', 'selected_ids': [item.message.message_id for item in ordinary_3_exposures]},
    {'case': 'ordinary 4, no leader ties', 'selected_ids': [item.message.message_id for item in ordinary_4_exposures]},
    {'case': 'press bypasses ties', 'selected_ids': [item.message.message_id for item in press_without_tie]},
    {'case': 'leader-to-leader inactive', 'selected_ids': [item.message.message_id for item in leader_to_leader]},
])
boundary_checks

## 6. Exploratory Stochastic Experiment

The experiment compares an equal-rate null with the provisional role-differentiated scenario. Each scenario has 5,000 independent selection trials for every recipient. Outputs are the press-delivery indicator and number of leader messages received per recipient-trial. Fixed four-standard-error tolerances, with a small numerical floor, are declared before inspecting the rates.

In [ ]:
def simulate_scenario(scenario_name, probabilities, seed):
    selector = make_selector(probabilities)
    rng = np.random.default_rng(seed)
    rows = []
    for trial in range(RUN['trials_per_scenario']):
        for recipient_id, recipient_kind in RECIPIENT_KIND_BY_ID.items():
            exposures = selector(
                recipient_id, MESSAGE_POOL, NETWORK, CONTEXT, rng
            )
            producer_ids = [item.message.producer_id for item in exposures]
            rows.append({
                'scenario': scenario_name,
                'trial': trial,
                'recipient_id': recipient_id,
                'recipient_kind': recipient_kind.value,
                'press_received': int(PRESS_ID in producer_ids),
                'leader_messages_received': sum(
                    ORIGINATOR_KIND_BY_ID[producer_id] is OriginatorKind.LEADER
                    for producer_id in producer_ids
                ),
            })
    return pd.DataFrame(rows)

trial_frames = []
for scenario_index, (scenario_name, probabilities) in enumerate(
    RUN['press_scenarios'].items()
):
    trial_frames.append(simulate_scenario(
        scenario_name, probabilities, RUN['seed'] + scenario_index
    ))
trials = pd.concat(trial_frames, ignore_index=True)
trials.head()

In [ ]:
press_rates = (
    trials.groupby(['scenario', 'recipient_kind'], as_index=False)
    .agg(observed_press_rate=('press_received', 'mean'), n=('press_received', 'size'))
)
press_rates['expected_press_rate'] = press_rates.apply(
    lambda row: RUN['press_scenarios'][row['scenario']][row['recipient_kind']],
    axis=1,
)
for row in press_rates.itertuples(index=False):
    standard_error = math.sqrt(
        row.expected_press_rate * (1.0 - row.expected_press_rate) / row.n
    )
    assert abs(row.observed_press_rate - row.expected_press_rate) <= max(
        0.01, 4 * standard_error
    )

expected_leader_exposure = {
    recipient_id: (
        len(NETWORK.eligible_producers(recipient_id))
        if kind is RecipientKind.ORDINARY
        else 0
    )
    for recipient_id, kind in RECIPIENT_KIND_BY_ID.items()
}
leader_exposure = (
    trials.groupby(['scenario', 'recipient_id'], as_index=False)
    .agg(observed_leader_messages=('leader_messages_received', 'mean'))
)
leader_exposure['expected_leader_messages'] = leader_exposure['recipient_id'].map(
    expected_leader_exposure
)
assert (
    leader_exposure['observed_leader_messages']
    == leader_exposure['expected_leader_messages']
).all()

display(press_rates.round(4))
display(leader_exposure)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
markers = {'leader': 'o', 'ordinary': 's'}
for recipient_kind, group in press_rates.groupby('recipient_kind'):
    axes[0].scatter(
        group['expected_press_rate'],
        group['observed_press_rate'],
        marker=markers[recipient_kind],
        s=60,
        label=recipient_kind,
    )
axes[0].plot([0, 1], [0, 1], color='black', linewidth=0.8, linestyle='--')
axes[0].set(
    title='Press delivery: observed vs expected',
    xlabel='Configured probability',
    ylabel='Observed delivery rate',
    xlim=(-0.02, 1.02),
    ylim=(-0.02, 1.02),
)
axes[0].legend(frameon=False)

network_pattern = leader_exposure.query(
    "scenario == 'role-differentiated'"
).sort_values('recipient_id')
axes[1].bar(
    network_pattern['recipient_id'].astype(str),
    network_pattern['observed_leader_messages'],
    color='#4c78a8',
)
axes[1].scatter(
    network_pattern['recipient_id'].astype(str),
    network_pattern['expected_leader_messages'],
    color='black',
    marker='_',
    s=220,
    label='expected from ties',
)
axes[1].set(
    title='Leader exposure follows incoming ties',
    xlabel='Recipient ID',
    ylabel='Mean leader messages per trial',
)
axes[1].legend(frameon=False)
fig.tight_layout()
plt.show()

## 7. Results and Conditional Interpretation

The next cell reports the observed V1 outcomes. All conclusions remain conditional on the fixed synthetic network, generated pool, independent Bernoulli press access, and deterministic leader-tie delivery.

In [ ]:
differentiated = press_rates.query(
    "scenario == 'role-differentiated'"
).set_index('recipient_kind')
ordinary_pattern = network_pattern.set_index('recipient_id')
display(Markdown(
    f"""
- **Role-dependent press access:** In the differentiated scenario, configured/observed rates are `0.8`/`{differentiated.loc['leader', 'observed_press_rate']:.3f}` for leaders and `0.2`/`{differentiated.loc['ordinary', 'observed_press_rate']:.3f}` for ordinary recipients.
- **Tie-dependent leader access:** Ordinary recipients 2, 3, and 4 receive `{ordinary_pattern.loc[2, 'observed_leader_messages']:.0f}`, `{ordinary_pattern.loc[3, 'observed_leader_messages']:.0f}`, and `{ordinary_pattern.loc[4, 'observed_leader_messages']:.0f}` leader messages per trial, matching their one, two, and zero incoming leader ties.
- **Inactive relation:** Leader recipients receive zero leader messages even when the deterministic boundary check inserts a leader-to-leader tie.
- **Highest completed level:** V1 isolated message-selection behavior under fixed generated boundaries.
- **Supports:** The package implementation realizes the specified separation between role-dependent press access and network-dependent leader access while remaining stance-blind.
- **Does not support:** The empirical truth of the selected probabilities, deterministic tie transmission, independence assumptions, network topology, or any claim about belief change or population opinion.
"""
))

## 8. Disposition and Change Impact

- **Disposition:** Retain as a V1 candidate for researcher review and later paired coupling; this is not yet empirical validation or full-model integration.
- **Active selection relations:** press→leader, press→ordinary, and leader→ordinary.
- **Consciously inactive relation:** leader→leader; the broader source-recipient aggregation interface still retains its four-cell relation matrix.
- **Affected mechanism:** `opleader` message selection and this probe only.
- **Unaffected:** Baseline selector, production mechanisms, source-recipient aggregation weights, Beta priors, opinion updating, and network updating.
- **Deferred alternatives:** probabilistic leader-tie delivery, repeated-exposure limits, attention capacity, feed ranking, homophilous selection, leader-to-leader diffusion, and press outlets with heterogeneous reach.
- **Required later check:** V2 production→selection coupling should verify message-pool timing and IDs; V2 selection→aggregation coupling should verify that recipient and originator identities survive unchanged.